### Text processing with AWK
[LabEx article](https://labex.io/tutorials/linux-linux-awk-command-with-practical-examples-422567)

Processes the file one line at a time.
Analyze log files, a common task for system administrators and data analysts. AWK is a powerful tool for processing structured text data in Linux, allowing you to extract, filter, and transform information efficiently.

Imagine you are a junior system administrator tasked with analyzing server logs to identify potential security threats and performance issues. The `awk` command will be your primary tool for this task, enabling you to quickly sift through large log files and extract meaningful insights.

### Examining the Log File

In [ ]:
labex:project/ $ head -n 5 server_logs.txt 
2023-08-02 06:55:10 192.168.1.174 POST /login.php 200
2023-08-02 03:09:39 192.168.1.139 GET /upload.php 200
2023-08-01 19:43:49 192.168.1.120 GET /static/style.css 200
2023-08-02 00:30:56 192.168.1.100 GET /index.html 200
2023-08-01 23:56:00 192.168.1.165 POST /index.html 200

This log file contains information about server requests, including the date and time, IP address, HTTP method, requested resource, and status code.

---

### Basic AWK Usage - Printing Specific Fields

Now that we've seen the structure of our log file, let's use AWK to extract specific information.

By default, AWK **splits each line into fields based on whitespace**. We can refer to these fields using `$1`, `$2`, etc., where `$1` is the first field, `$2` is the second, and so on.

Let's extract the IP addresses (the third field) from our log file:

In [ ]:
labex:project/ $ awk '{print $3}' server_logs.txt | head -n 5 
192.168.1.174
192.168.1.139
192.168.1.120
192.168.1.100
192.168.1.165

In this command:

- `awk '{print $3}'` tells AWK to print the third field of each line.
- We pipe (`|`) the output to `head -n 5` to limit the display to the first 5 lines.

Now, let's print both the IP address and the requested resource:

In [ ]:
labex:project/ $ awk '{print $3, $5}' server_logs.txt | head -n 5
192.168.1.174 /login.php
192.168.1.139 /upload.php
192.168.1.120 /static/style.css
192.168.1.100 /index.html
192.168.1.165 /index.html

Here, we're printing the third field (IP address) and the fifth field (requested resource) for each line.

---

### Filtering Log Entries
One of AWK's strengths is its ability to **filter data based on conditions**. Let's use this feature to find all POST requests in our log file, as these might be more security-sensitive than GET requests.

Run the following command:

In [ ]:
labex:project/ $ awk '$4 == "POST" {print $0}' server_logs.txt
2023-08-01 17:22:16 192.168.1.106 POST /products.html 301
2023-08-02 06:19:41 192.168.1.182 POST /upload.php 200
2023-08-01 09:55:46 192.168.1.111 POST /api/data 200
2023-08-02 00:48:12 192.168.1.165 POST /login.php 200
2023-08-02 04:14:02 192.168.1.124 POST /services.html 500
2023-08-01 10:24:01 192.168.1.104 POST /products.html 200
2023-08-02 00:48:22 192.168.1.113 POST /upload.php 200

This command may print hundreds of lines because the sample file contains 5,000 log entries. If you only want to inspect a manageable sample while learning, add `| head -n 10`:

In [ ]:
labex:project/ $ awk '$4 == "POST" {print $0}' server_logs.txt | head -n 10
2023-08-02 06:55:10 192.168.1.174 POST /login.php 200
2023-08-01 23:56:00 192.168.1.165 POST /index.html 200
2023-08-02 01:36:33 192.168.1.197 POST /products.html 200
2023-08-01 22:39:22 192.168.1.172 POST /submit_form.php 200
2023-08-01 16:07:55 192.168.1.118 POST /contact.html 200
2023-08-02 01:22:00 192.168.1.144 POST /index.html 301
2023-08-01 10:49:33 192.168.1.142 POST /api/data 200
2023-08-02 03:12:33 192.168.1.102 POST /login.php 200
2023-08-02 00:28:35 192.168.1.191 POST /about.html 200
2023-08-02 04:39:31 192.168.1.163 POST /submit_form.php 200

The verification still accepts the plain `awk` command, so use whichever version helps you read the output more comfortably.

Let's break down this command's syntax to understand how AWK filtering works:

1. `$4 == "POST"` - This is a pattern or condition that AWK evaluates for each line:
    
    - `$4` refers to the fourth field in the current line (in our log file, this is the HTTP method)
    - `==` is the equality operator that checks if two values are equal
    - `"POST"` is the string we're comparing against
2. `{print $0}` - This is the action AWK performs when the condition is true:
    
    - The curly braces `{}` enclose the action
    - `print` is the command to output text
    - `$0` represents the entire current line (all fields)

The command structure follows the AWK pattern: `condition {action}`. AWK reads each line, and if the condition evaluates to true, it performs the action. If no condition is specified (as in our earlier examples), the action is performed for every line.

You should see output similar to this:

In [ ]:
2023-08-01 08:17:30 192.168.1.102 POST /login.php 302
2023-08-01 09:23:45 192.168.1.110 POST /submit_form.php 200
2023-08-01 10:45:12 192.168.1.115 POST /upload.php 500

Now, let's find all requests that resulted in a 404 (Not Found) status:

In [ ]:
labex:project/ $ awk '$6 == "404" {print $1, $2, $5}' server_logs.txt
...
2023-08-01 08:38:17 /submit_form.php
2023-08-02 03:27:03 /upload.php
2023-08-01 16:51:18 /about.html
2023-08-01 10:01:02 /products.html
2023-08-01 14:54:04 /about.html
2023-08-01 23:48:46 /submit_form.php
2023-08-02 07:39:04 /static/style.css
2023-08-01 12:58:39 /upload.php

This command follows the same pattern but with different values:

- The condition `$6 == "404"` checks if the sixth field (status code) equals 404
- The action `{print $1, $2, $5}` prints only specific fields:
    - `$1` - First field (date)
    - `$2` - Second field (time)
    - `$5` - Fifth field (requested resource)

This selective printing allows you to focus on just the information you need.

You can combine multiple conditions using logical operators:

- `&&` for AND (both conditions must be true)
- `||` for OR (at least one condition must be true)
- `!` for NOT (negates a condition)

For example, to find all POST requests that resulted in an error (status code >= 400):

In [ ]:
labex:project/ $ awk '$4 == "POST" && $6 >= 400 {print $0}' server_logs.txt
2023-08-01 10:28:06 192.168.1.113 POST /static/style.css 403
2023-08-01 08:49:54 192.168.1.136 POST /index.html 400
2023-08-02 04:09:55 192.168.1.183 POST /api/data 400
2023-08-01 18:07:43 192.168.1.139 POST /services.html 403
2023-08-01 12:27:59 192.168.1.168 POST /upload.php 403
2023-08-02 03:01:30 192.168.1.169 POST /contact.html 400
2023-08-01 09:47:31 192.168.1.198 POST /contact.html 500
2023-08-02 04:14:02 192.168.1.124 POST /services.html 500

These filters can help you quickly identify potential issues or suspicious activities in your server logs.

---

### Counting and Summarizing Data
AWK is excellent for **counting occurrences and summarizing data**. 

Let's use it to count the number of requests for each HTTP status code.

Run this command:

In [ ]:
labex:project/ $ awk '{count[$6]++} END {for (code in count) print code, count[code]}' server_logs.txt | sort -n    
200 3546
301 248
302 254
304 226
400 268
403 100
404 226
500 132

This command is more complex, so let's break it down step by step:

1. `{count[$6]++}` - This is the main action performed for each line:
    
    - `count` is an array (associative array or dictionary) we're creating
    - `[$6]` uses the value of the 6th field (status code) as the array index/key
    - `++` is the increment operator, adding 1 to the current value
    - So for each line, we increment the counter for the specific status code found
2. `END {for (code in count) print code, count[code]}` - This is executed after processing all lines:
    
    - `END` is a special pattern that matches the end of the input
    - `{...}` contains the action to perform after all input is processed
    - `for (code in count)` is a loop that iterates through all keys in the `count` array
    - `print code, count[code]` prints each status code and its count
3. `| sort -n` - Pipes the output to the sort command, which sorts numerically

When AWK processes an array like `count[$6]++`, it automatically:

- Creates the array if it doesn't exist
- Creates a new element with value 0 if the key doesn't exist
- Then increments the value by 1

This summary quickly shows you the distribution of status codes in your log file.

Now, let's find the top 5 most frequently accessed resources:

In [ ]:
labex:project/ $ awk '{count[$5]++} END {for (resource in count) print count[resource], resource}' server_logs.txt | sort -rn | head -n 5
544 /services.html
543 /products.html
527 /api/data
521 /contact.html
497 /login.php

This command follows a similar pattern with a few changes:

1. `{count[$5]++}` - Counts occurrences of the 5th field (the requested resource)
2. `END {for (resource in count) print count[resource], resource}` - After processing all lines:
    - Prints the count first, followed by the resource
    - This order change facilitates numerical sorting by count
3. `| sort -rn` - Sorts numerically in reverse order (highest counts first)
4. `| head -n 5` - Limits output to the first 5 lines (top 5 results)

These AWK commands demonstrate the power of using arrays for counting and summarizing. You can adapt this pattern to count any field or combination of fields in your data.

For example, to count the number of requests per IP address:

In [ ]:
labex:project/ $ awk '{count[$3]++} END {for (ip in count) print ip, count[ip]}' server_logs.txt
...
192.168.1.119 47
192.168.1.132 54
192.168.1.133 46
192.168.1.134 47
192.168.1.135 57
192.168.1.136 49

This command counts how many times each IP address appears in `server_logs.txt`.

#### Parts of the command

- `awk`  
    Processes the file one line at a time.
    
- `server_logs.txt`  
    The input file being read.
    
- `{count[$3]++}`  
    Runs for every line:
    
    - `$3` means the third whitespace-separated field.
    - In this log format, the third field is the IP address.
    - `count[$3]` stores a counter for that IP address.
    - `++` increases the counter by 1.
    - If the IP has not appeared before, AWK starts its count at `0`, then increments it to `1`.
- `END { ... }`  
    Runs once after all lines have been processed.
    
- `for (ip in count)`  
    Loops through every unique IP address stored in the `count` array.
    
    - `ip` represents the current IP address.
- `print ip, count[ip]`  
    Prints the IP address followed by the number of times it appeared.

To count requests by both method and status:

In [ ]:
labex:project/ $ awk '{key=$4"-"$6; count[key]++} END {for (k in count) print k, count[k]}' server_logs.txt           
PUT-404 9
DELETE-500 8
GET-403 69
GET-200 2414
PUT-500 4
GET-404 158
...

This command counts how often each **combination of HTTP method and status code** appears in `server_logs.txt`.

#### Important parts

- `awk`  
    Reads the file one line at a time.
    
- `server_logs.txt`  
    The file being processed.
    
- `$4`  
    Refers to the fourth whitespace-separated field, which is the HTTP method, such as `GET` or `POST`.
    
- `$6`  
    Refers to the sixth field, which is the HTTP status code, such as `200` or `404`.
    
- `key=$4"-"$6`  
    Combines the method and status code into one key, separated by a hyphen.
    
    For example:

In [ ]:
POST 200

becomes:

In [ ]:
POST-200

- `count[key]++`  
    Uses the combined key as an array index and increases its counter by 1. Each unique method/status combination gets its own count.
    
- `END { ... }`  
    Runs after every line in the file has been processed.
    
- `for (k in count)`  
    Loops through all the unique combinations stored in the `count` array.
    
- `print k, count[k]`  
    Prints each combination and its total count.
    

Example output:

In [ ]:
POST-200 42
POST-404 7
GET-200 18
GET-500 2

Overall, the command groups log entries by both HTTP method and status code, then reports how many times each combination occurred.

---

### Creating a Simple Report

For our final task, let's create a simple HTML report summarizing some key information from our log file. We'll use an AWK script stored in a separate file for this more complex operation.

This step combines several AWK ideas from earlier sections:

- counters such as `total++`
- arrays such as `ip_count[$3]++`
- an `END` block that prints the final summary

If the script feels long at first glance, focus on one block at a time. You do not need to memorize the whole file before running it.

First, create a file named `log_report.awk` with the following content:

> Tips: Copy the content below and paste it into your terminal to create the file.

In [ ]:
cat << 'EOF' > log_report.awk
BEGIN {
    print "<html><body>"
    print "<h1>Server Log Summary</h1>"
    total = 0
    errors = 0
}

{
    total++
    if ($6 >= 400) errors++
    ip_count[$3]++
    resource_count[$5]++
}

END {
    print "<p>Total requests: " total "</p>"
    print "<p>Error rate: " (errors/total) * 100 "%</p>"
    
    print "<h2>Top 5 IP Addresses</h2>"
    print "<ul>"
    for (ip in ip_count) {
        top_ips[ip] = ip_count[ip]
    }
    n = asort(top_ips, sorted_ips, "@val_num_desc")
    for (i = 1; i <= 5 && i <= n; i++) {
        for (ip in ip_count) {
            if (ip_count[ip] == sorted_ips[i]) {
                print "<li>" ip ": " ip_count[ip] " requests</li>"
                delete ip_count[ip]
                break
            }
        }
    }
    print "</ul>"
    
    print "<h2>Top 5 Requested Resources</h2>"
    print "<ul>"
    for (resource in resource_count) {
        top_resources[resource] = resource_count[resource]
    }
    n = asort(top_resources, sorted_resources, "@val_num_desc")
    for (i = 1; i <= 5 && i <= n; i++) {
        for (resource in resource_count) {
            if (resource_count[resource] == sorted_resources[i]) {
                print "<li>" resource ": " resource_count[resource] " requests</li>"
                delete resource_count[resource]
                break
            }
        }
    }
    print "</ul>"
    
    print "</body></html>"
}
EOF

Let's understand this AWK script section by section:

1. BEGIN Block: Executes before processing any input lines

In [ ]:
BEGIN {
    print "<html><body>"  # Start HTML structure
    print "<h1>Server Log Summary</h1>"
    total = 0  # Initialize counter for total requests
    errors = 0  # Initialize counter for error requests
}

2. Main Processing Block: Executes for each line of the input file

In [ ]:
{
    total++  # Increment total request counter
    if ($6 >= 400) errors++  # Count error responses (status codes >= 400)
    ip_count[$3]++  # Count requests by IP address (field 3)
    resource_count[$5]++  # Count requests by resource (field 5)
}

3. END Block: Executes after processing all input lines

In [ ]:
END {
    # Print summary statistics
    print "<p>Total requests: " total "</p>"
    print "<p>Error rate: " (errors/total) * 100 "%</p>"

    # Process and print top 5 IP addresses
    # ...

    # Process and print top 5 requested resources
    # ...

    print "</body></html>"  # End HTML structure
}

Before moving on, notice the overall flow:

1. `BEGIN` prints the opening HTML tags and initializes counters.
2. The middle block processes each log line and updates totals.
3. `END` prints the final report after every line has been analyzed.

Let's examine the sorting logic for the top IPs (the resources section works the same way):

In [ ]:
# Copy the counts to a new array for sorting
for (ip in ip_count) {
    top_ips[ip] = ip_count[ip]
}

# Sort the array by value in descending order
n = asort(top_ips, sorted_ips, "@val_num_desc")

# Print the top 5 entries
for (i = 1; i <= 5 && i <= n; i++) {
    # Find the original IP that matches this count
    for (ip in ip_count) {
        if (ip_count[ip] == sorted_ips[i]) {
            print "<li>" ip ": " ip_count[ip] " requests</li>"
            delete ip_count[ip]
            break
        }
    }
}

In this script:

- The `asort()` function sorts the array
- `"@val_num_desc"` is a special argument that tells it to sort numerically by value in descending order
- The nested loops find and print the top 5 entries

You can think of the nested loops like this:

- the first loop decides which counts belong in the top 5
- the second loop finds which IP address or resource produced each count
- after printing one match, the script deletes that key so equal counts do not duplicate the same entry

That lookup pattern is more advanced than the previous steps, so it is normal if this is the first part of the lab that feels like real scripting instead of a one-line command.

Now, let's run our AWK script to generate the report:

In [ ]:
awk -f log_report.awk server_logs.txt > log_report.html

The `-f` option tells AWK to read the script from the specified file:

- `-f log_report.awk` - Reads the AWK script from the file `log_report.awk`
- `server_logs.txt` - Processes this file using the script
- `> log_report.html` - Redirects the output to the file `log_report.html`

You can view the contents of the report using the `cat` command:

In [ ]:
cat log_report.html

If the HTML output feels hard to scan in the terminal, preview just the first part first:

In [ ]:
head -n 15 log_report.html

This report provides a summary of total requests, error rate, top 5 IP addresses, and top 5 requested resources. In a real-world scenario, you could open this HTML file in a web browser for a formatted view.

The approach we've used in this script demonstrates how AWK can be used for more complex data analysis tasks. You can extend this script to include additional statistics or different visualizations based on your specific needs.

---

### Summary
Congratulations! You've completed this lab on using the AWK command for log analysis. Let's recap what you've learned:

1. Basic AWK usage: Printing specific fields from a structured text file.
2. Filtering data: Using conditions in AWK to select specific log entries.
3. Counting and summarizing: Using AWK to generate statistics from log data.
4. Creating reports: Writing more complex AWK scripts to generate formatted reports.

These skills will be invaluable for analyzing log files, processing data, and generating reports in your future work as a system administrator or data analyst.

Here are some additional AWK parameters and features we didn't cover in this lab:

- `-F`: Specifies a field separator other than whitespace.
- `-v`: Assigns a value to a variable.
- `NR`: A built-in variable representing the current record number.
- `NF`: A built-in variable representing the number of fields in the current record.
- `BEGIN` and `END` blocks: Special patterns for initialization and finalization.
- Built-in functions: Mathematical functions, string functions, and more.

Remember, practice is key to mastering AWK. Try modifying the commands and scripts from this lab to analyze different aspects of the log file or to process other types of structured text data.